Temp Views & Global Views

In [0]:
from pyspark.sql import *
from pyspark.sql.types import *
from pyspark.sql.functions import *
df=spark.read.format("csv")\
    .option("header",True)\
    .option("inferSchema","true")\
        .load("/Volumes/bronze_layer/demo/pyspark_demo_vol/customers-100.csv")
df.display()

Creating Temp View

In [0]:
df.createOrReplaceTempView("Cust_Temp")

In [0]:
%sql
select * from Cust_Temp where Index=20;

Index,Customer Id,First Name,Last Name,Company,City,Country,Phone 1,Phone 2,Email,Subscription Date,Website
20,0F60FF3DdCd7aB0,Joanna,Kirk,Mays-Mccormick,Jamesshire,French Polynesia,(266)131-7001x711,(283)312-5579x11543,tuckerangie@salazar.net,2021-09-24,https://www.camacho.net/


In [0]:
df.createOrReplaceGlobalTempView("Cust_Temp_Global")

In [0]:
%sql
Select * from Cust_Temp_Global

In [0]:
spark.sql("CREATE TABLE IF NOT EXISTS bronze_layer.demo.customer_temp")

DataFrame[]

##Joins

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
# Sample data: Employees
emp_data = [
    (1, "Alice", 10),
    (2, "Bob", 20),
    (3, "Charlie", 30),
    (4, "David", 40),   # Dept 40 doesn't exist in dept_data
    (5, "Eve", None)    # Null department
]
emp_schema = ["emp_id", "emp_name", "dept_id"]
df_emp = spark.createDataFrame(emp_data, schema=emp_schema)

# Sample data: Departments
dept_data = [
    (10, "HR"),
    (20, "Engineering"),
    (30, "Marketing"),
    (50, "Finance")     # No employees currently match this
]
dept_schema = ["dept_id", "dept_name"]
df_dept = spark.createDataFrame(dept_data, schema=dept_schema)

# Register tables as temporary views to enable raw SQL queries
df_emp.createOrReplaceTempView("employees")
df_dept.createOrReplaceTempView("departments")

print("Tables successfully initialized as 'employees' and 'departments' views.")


Tables successfully initialized as 'employees' and 'departments' views.


In [0]:
spark.sql("Select e.*,d.* from employees e INNER JOIN departments d on e.dept_id=d.dept_id").show()

+------+--------+-------+-------+-----------+
|emp_id|emp_name|dept_id|dept_id|  dept_name|
+------+--------+-------+-------+-----------+
|     1|   Alice|     10|     10|         HR|
|     2|     Bob|     20|     20|Engineering|
|     3| Charlie|     30|     30|  Marketing|
+------+--------+-------+-------+-----------+



##Upsert

In [0]:
%sql
CREATE TABLE IF NOT EXISTS default.employees (
    emp_id INT,
    emp_name STRING,
    department STRING,
    salary INT,
    last_updated TIMESTAMP
) 
USING DELTA;


In [0]:
%sql
-- Insert initial records
INSERT INTO default.employees VALUES
    (1, 'Alice', 'Engineering', 90000, TIMESTAMP '2026-06-01 10:00:00'),
    (2, 'Bob', 'Sales', 70000, TIMESTAMP '2026-06-01 10:00:00'),
    (3, 'Charlie', 'Marketing', 65000, TIMESTAMP '2026-06-01 10:00:00');


num_affected_rows,num_inserted_rows
3,3


In [0]:
%sql
-- Create a staging view representing an incoming batch of incremental data
CREATE OR REPLACE TEMPORARY VIEW incoming_employee_updates AS
VALUES
    (1, 'Alice', 'Data Science', 110000, TIMESTAMP '2026-06-05 14:00:00'), -- Update
    (2, 'Bob', 'Sales', 70000, TIMESTAMP '2026-06-01 10:00:00'),          -- No-op
    (4, 'David', 'HR', 55000, TIMESTAMP '2026-06-05 14:00:00');          -- New Insert


In [0]:
%sql
MERGE INTO default.employees AS target
USING incoming_employee_updates AS source
ON target.emp_id = source.col1
WHEN MATCHED AND target.last_updated < source.col5 THEN 
  UPDATE SET 
    target.emp_name = source.col2,
    target.department = source.col3,
    target.salary = source.col4,
    target.last_updated = source.col5
WHEN NOT MATCHED THEN 
  INSERT (emp_id, emp_name, department, salary, last_updated) 
  VALUES (source.col1, source.col2, source.col3, source.col4, source.col5)

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
2,1,0,1


##Explain

In [0]:
%sql
explain select * from default.employees

plan
"== Physical Plan == PhotonResultStage +- PhotonColumnarToRow +- PhotonScan parquet workspace.default.employees[emp_id#12682,emp_name#12683,department#12684,salary#12685,last_updated#12686] DataFilters: [], DictionaryFilters: [], Format: parquet, Location: PreparedDeltaFileIndex(1 paths)[s3://databricks-aterhwtdb8q47wjvdsx1fy-cloud-storage-bucket/unity..., OptionalDataFilters: [], PartitionFilters: [], ReadSchema: struct, RequiredDataFilters: [] == Photon Explanation == The query is fully supported by Photon. == Optimizer Statistics (table names per statistics state) == missing = partial = full = employees"


Read Files Using SparkSQl

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW CUSTOMER_CSV
USING CSV
OPTIONS(
    PATH "/Volumes/bronze_layer/demo/external_volumne/csv_append/part-00002-tid-2410573440913868416-670875a9-8149-4e19-ada3-e3c99b3cf0ba-142-1-c000.csv",
    header "true",
    inferSchema "true"
);

select * from CUSTOMER_CSV

Item,Price,Discount_Pct,Final_Price
Monitor,300.0,0,300.0


In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW CUSTOMER_JSON
USING JSON
OPTIONS(
    PATH "/Volumes/bronze_layer/demo/external_volumne/json_append/part-00000-tid-1356917729146289525-bda138a9-0ecb-4144-9a3d-65af5ce47694-153-1-c000.json",
    header "true",
    inferSchema "true"
);

select * from CUSTOMER_JSON

Discount_Pct,Final_Price,Item,Price
10,1080.0,Laptop,1200.0


dynamic dataframes passing

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
# Sample data: Employees
emp_data = [
    (1, "Alice", 10),
    (2, "Bob", 20),
    (3, "Charlie", 30),
    (4, "David", 40),   # Dept 40 doesn't exist in dept_data
    (5, "Eve", None)    # Null department
]
emp_schema = ["emp_id", "emp_name", "dept_id"]
df_emp = spark.createDataFrame(emp_data, schema=emp_schema)

In [0]:
display(spark.sql("SELECT * FROM {df_jas}",df_jas=df_emp))

emp_id,emp_name,dept_id
1,Alice,10
2,Bob,20
3,Charlie,30
4,David,40
5,Eve,null


File Format Connector

In [0]:
display(spark.sql("SELECT * FROM json.`/Volumes/bronze_layer/demo/external_volumne/json_append/part-00000-tid-1356917729146289525-bda138a9-0ecb-4144-9a3d-65af5ce47694-153-1-c000.json`"))

Discount_Pct,Final_Price,Item,Price
10,1080.0,Laptop,1200.0
